# Stage 0: which Qwen3.5-4B layer makes extraction readable?

The backbone is frozen. One span+null probe head per layer is trained on the same batches, then every head
predicts on held-out sets. The question: can a frozen base already locate the answer span and detect absent
fields, and at which depth? This decides whether we move on to heads-only training and then LoRA.

Scope: single-span field types (`text`, `number`, `date`, `time`, `datetime`, `duration`) and null detection.
Compute fields only train the null head; list, estimate and forecast fields are left out (stage 1+).

**Run:** Runtime → Change runtime type → A100. Set `REPO_URL` below, then Run all.
`DRY_RUN=True` runs everything on CPU with a tiny random backbone and small data (a smoke test).

In [ ]:
# --- config ---
REPO_URL = "https://github.com/pherber3/extraction-model.git"
BRANCH = "main"
DRY_RUN = False
OUT_DIR = "/content/stage0_results"
INCLUDE_HEADS = False   # True: also put the probe weights in the zip (~1 GB for 32 layers)
# Data sizes, epochs and layers: see extraction/stage0.py (Stage0Config); override below if needed.

In [ ]:
# --- setup: clone/pull the repo, install what Colab lacks ---
import os, subprocess, sys
REPO_DIR = "/content/extraction-model"
if os.path.isdir("/content"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    pkgs = ["transformers>=5.20", "datasets"] + ([] if DRY_RUN else ["flash-linear-attention"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
else:  # running locally from a repo checkout
    REPO_DIR = os.path.abspath("..")
    OUT_DIR = os.path.join(REPO_DIR, "results", "stage0")
sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())

## Run: data, one probe per layer, evaluation of every layer

In [ ]:
from extraction.stage0 import Stage0Config, run
config = Stage0Config(dry_run=DRY_RUN, out_dir=OUT_DIR)
result = run(config)

## Save results

In [ ]:
import shutil, tempfile
# Zip the results (tables, per-field rows, train log); probe weights only if INCLUDE_HEADS.
staging = tempfile.mkdtemp()
shutil.copytree(OUT_DIR, os.path.join(staging, "stage0_results"),
                ignore=None if INCLUDE_HEADS else shutil.ignore_patterns("*.pt"))
archive = shutil.make_archive(OUT_DIR, "zip", staging)
print(f"wrote {archive} ({os.path.getsize(archive) / 2**20:.1f} MiB)")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass  # local run: the zip is next to OUT_DIR